# Validación del Pipeline ETL con SQLite

## Objetivo

Este notebook verifica la ejecución del pipeline ETL de datos clínicos y la persistencia de los datos transformados en una base de datos SQLite.

Se validan la cantidad de registros, la integridad de la clave primaria, la conservación del esquema SQL y la repetibilidad del proceso.
  

In [1]:
from pathlib import Path
import sys
import sqlite3
import pandas as pd

# Buscar la carpeta que contiene main.py
ruta_actual = Path.cwd()
ruta_proyecto = ruta_actual

while not (ruta_proyecto / "main.py").exists():
    if ruta_proyecto.parent == ruta_proyecto:
        raise FileNotFoundError(
            "No se encontró main.py en esta carpeta ni en sus directorios superiores."
        )
    ruta_proyecto = ruta_proyecto.parent

# Agregar la raíz del proyecto a las rutas de importación
if str(ruta_proyecto) not in sys.path:
    sys.path.insert(0, str(ruta_proyecto))

ruta_db = (
    ruta_proyecto
    / "data"
    / "processed"
    / "clinical_trials.db"
)

print("Raíz del proyecto:", ruta_proyecto)
print("Base de datos:", ruta_db)
print("Base de datos existente:", ruta_db.exists())


Raíz del proyecto: C:\Users\luisi\Desktop\Bioinformatica\Data Science\Data science II\clinical_trial_pipeline Pre entrega 5\clinical-trial-etl
Base de datos: C:\Users\luisi\Desktop\Bioinformatica\Data Science\Data science II\clinical_trial_pipeline Pre entrega 5\clinical-trial-etl\data\processed\clinical_trials.db
Base de datos existente: False


In [2]:
from main import main

main()

=== INICIO DEL PIPELINE ===

ETAPA 1: EXTRACCIÓN
Dataset cargado: 1000 filas y 11 columnas
Registros extraídos: 1000

ETAPA 2: TRANSFORMACIÓN
Nombres de columnas normalizados.
Fecha de inscripción convertida correctamente.
Duplicados eliminados: 0
Filas eliminadas por datos esenciales faltantes: 0
Transformación completada.
Filas finales: 1000
Columnas finales: 11
Valores nulos restantes: 0
Registros transformados: 1000

ETAPA 3: PERSISTENCIA
Base de datos: C:\Users\luisi\Desktop\Bioinformatica\Data Science\Data science II\clinical_trial_pipeline Pre entrega 5\clinical-trial-etl\data\processed\clinical_trials.db
Tabla: clinical_trials
Registros almacenados: 1000

=== PIPELINE FINALIZADO ===


In [3]:
from src.extract import extract_data
from src.transform import transform_data

ruta_csv = (
    ruta_proyecto
    / "data"
    / "raw"
    / "synthetic_clinical_trial_data.csv"
)

df_original = extract_data(str(ruta_csv))
df_limpio = transform_data(df_original)

print("Columnas originales:")
print(df_original.columns.tolist())

print("\nColumnas después de transformar:")
print(df_limpio.columns.tolist())

print("\nDimensiones originales:", df_original.shape)
print("Dimensiones transformadas:", df_limpio.shape)

Dataset cargado: 1000 filas y 11 columnas
Nombres de columnas normalizados.
Fecha de inscripción convertida correctamente.
Duplicados eliminados: 0
Filas eliminadas por datos esenciales faltantes: 0
Transformación completada.
Filas finales: 1000
Columnas finales: 11
Valores nulos restantes: 0
Columnas originales:
['Subject_ID', 'Site_ID', 'Age', 'Gender', 'Enrollment_Date', 'Treatment_Group', 'Adverse_Events', 'Dropout', 'Systolic_BP', 'Diastolic_BP', 'Cholesterol_Level']

Columnas después de transformar:
['subject_id', 'site_id', 'age', 'gender', 'enrollment_date', 'treatment_group', 'adverse_events', 'dropout', 'systolic_bp', 'diastolic_bp', 'cholesterol_level']

Dimensiones originales: (1000, 11)
Dimensiones transformadas: (1000, 11)


In [4]:
with sqlite3.connect(ruta_db) as conexion:
    total = conexion.execute(
        "SELECT COUNT(*) FROM clinical_trials"
    ).fetchone()[0]

    total_ids = conexion.execute(
        """
        SELECT COUNT(DISTINCT subject_id)
        FROM clinical_trials
        """
    ).fetchone()[0]

    ids_nulos = conexion.execute(
        """
        SELECT COUNT(*)
        FROM clinical_trials
        WHERE subject_id IS NULL
        """
    ).fetchone()[0]

    muestra = pd.read_sql_query(
        "SELECT * FROM clinical_trials LIMIT 5",
        conexion
    )

print("Total de registros:", total)
print("subject_id únicos:", total_ids)
print("subject_id nulos:", ids_nulos)

display(muestra)

assert total == 1000, f"Se esperaban 1000 registros; se encontraron {total}."
assert total_ids == total, "Hay subject_id duplicados."
assert ids_nulos == 0, "Hay subject_id nulos."

print("Persistencia validada correctamente.")
  

Total de registros: 1000
subject_id únicos: 1000
subject_id nulos: 0


,subject_id,site_id,age,gender,enrollment_date,treatment_group,adverse_events,dropout,systolic_bp,diastolic_bp,cholesterol_level
0,1,49,54,Male,2022-01-01,Drug A,0,0,117,74,229
1,2,37,44,Male,2022-01-02,Placebo,1,0,111,57,173
2,3,1,58,Male,2022-01-03,Drug A,0,1,122,89,220
3,4,25,48,Male,2022-01-04,Drug B,0,0,122,85,175
4,5,10,57,Female,2022-01-05,Drug A,2,0,105,90,185


Persistencia validada correctamente.


In [5]:
main()

with sqlite3.connect(ruta_db) as conexion:
    total_final = conexion.execute(
        "SELECT COUNT(*) FROM clinical_trials"
    ).fetchone()[0]

print("Registros tras la segunda ejecución:", total_final)

assert total_final == 1000, "La segunda ejecución produjo un resultado inesperado."

print("Prueba de repetibilidad superada.")
  

=== INICIO DEL PIPELINE ===

ETAPA 1: EXTRACCIÓN
Dataset cargado: 1000 filas y 11 columnas
Registros extraídos: 1000

ETAPA 2: TRANSFORMACIÓN
Nombres de columnas normalizados.
Fecha de inscripción convertida correctamente.
Duplicados eliminados: 0
Filas eliminadas por datos esenciales faltantes: 0
Transformación completada.
Filas finales: 1000
Columnas finales: 11
Valores nulos restantes: 0
Registros transformados: 1000

ETAPA 3: PERSISTENCIA
Base de datos: C:\Users\luisi\Desktop\Bioinformatica\Data Science\Data science II\clinical_trial_pipeline Pre entrega 5\clinical-trial-etl\data\processed\clinical_trials.db
Tabla: clinical_trials
Registros almacenados: 1000

=== PIPELINE FINALIZADO ===
Registros tras la segunda ejecución: 1000
Prueba de repetibilidad superada.


In [6]:
with sqlite3.connect(ruta_db) as conexion:
    esquema = conexion.execute(
        """
        SELECT sql
        FROM sqlite_master
        WHERE type = 'table'
          AND name = 'clinical_trials'
        """
    ).fetchone()

assert esquema is not None, "No se encontró la tabla."
assert "PRIMARY KEY" in esquema[0].upper()
assert "CHECK" in esquema[0].upper()

print(esquema[0])
print("\nEsquema SQL validado correctamente.")
  

CREATE TABLE clinical_trials (
    subject_id INTEGER PRIMARY KEY,
    site_id INTEGER NOT NULL,
    age INTEGER,
    gender TEXT,
    enrollment_date TEXT,
    treatment_group TEXT,
    adverse_events INTEGER,
    dropout INTEGER CHECK (dropout IN (0, 1)),
    systolic_bp INTEGER,
    diastolic_bp INTEGER,
    cholesterol_level INTEGER
)

Esquema SQL validado correctamente.


In [7]:
import pandas as pd
import sqlite3

print("pandas:", pd.__version__)
print("SQLite:", sqlite3.sqlite_version)

pandas: 3.0.6
SQLite: 3.53.4
